## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 3: Matrix Factorization for Recommender Systems
## Task 5: Implementing SVD for Recommendations

Perform Singular Value Decomposition (SVD) on the user-item rating matrix to predict missing ratings.
Generate recommendations from reconstructed ratings and compare with user-based and item-based CF.

$$R \approx U \cdot \Sigma \cdot V^T$$

- $U$ = User latent factor matrix
- $\Sigma$ = Diagonal matrix of singular values
- $V^T$ = Item latent factor matrix

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'numpy', 'scipy', '-q'])

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
print('All imports successful!')

All imports successful!


### Step 1: Load Dataset and Prepare the User-Item Matrix

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_dir = "ml-latest-small"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print("Downloaded and extracted.")
else:
    print("Dataset already exists.")

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f"Movies: {len(movies)}, Ratings: {len(ratings)}, Users: {ratings['userId'].nunique()}")

Dataset already exists.
Movies: 9742, Ratings: 100836, Users: 610


In [3]:
# Build user-item rating matrix
rating_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
print(f"Rating matrix shape: {rating_matrix.shape}")
print(f"Sparsity: {rating_matrix.isna().sum().sum() / (rating_matrix.shape[0] * rating_matrix.shape[1]) * 100:.2f}%")

# Compute per-user mean ratings
user_means = rating_matrix.mean(axis=1)
print(f"\nUser mean ratings — min: {user_means.min():.2f}, max: {user_means.max():.2f}, avg: {user_means.mean():.2f}")

Rating matrix shape: (610, 9724)
Sparsity: 98.30%

User mean ratings — min: 1.27, max: 5.00, avg: 3.66


### Step 2: Normalize and Fill Missing Values

We normalize by subtracting each user's mean rating (mean-centering), then fill missing values with 0. This way, missing entries are treated as "neutral" relative to the user's baseline.

In [4]:
# Mean-center the ratings (subtract user mean)
rating_centered = rating_matrix.sub(user_means, axis=0)

# Fill NaN with 0 (neutral after centering)
rating_filled = rating_centered.fillna(0)

print(f"Centered matrix shape: {rating_filled.shape}")
print(f"Sample values (User 1, first 5 movies):")
print(rating_filled.iloc[0, :5])

Centered matrix shape: (610, 9724)
Sample values (User 1, first 5 movies):
movieId
1   -0.366379
2    0.000000
3   -0.366379
4    0.000000
5    0.000000
Name: 1, dtype: float64


### Step 3: Apply SVD

Factorize $R$ into $U \cdot \Sigma \cdot V^T$ using scipy's `svds` for truncated SVD (keeping top-k factors).

In [5]:
# Number of latent factors
k = 50

# Perform truncated SVD
U, sigma, Vt = svds(rating_filled.values, k=k)

# Convert sigma to diagonal matrix
sigma_diag = np.diag(sigma)

print(f"U shape: {U.shape}  (users x latent factors)")
print(f"Sigma shape: {sigma_diag.shape}  (latent factors x latent factors)")
print(f"Vt shape: {Vt.shape}  (latent factors x items)")
print(f"\nTop singular values: {sigma[-5:][::-1]}")

U shape: (610, 50)  (users x latent factors)
Sigma shape: (50, 50)  (latent factors x latent factors)
Vt shape: (50, 9724)  (latent factors x items)

Top singular values: [76.20046537 43.6224036  41.77917206 39.37050585 37.95619249]


### Step 4: Reconstruct the Approximate Rating Matrix

$$\hat{R} = U \cdot \Sigma_k \cdot V^T + \text{user\_means}$$

We add back the user means since we mean-centered earlier. Predictions are clipped to [0.5, 5.0].

In [6]:
# Reconstruct: R_hat = U * Sigma * Vt + user_means
predicted_centered = U @ sigma_diag @ Vt
predicted_ratings = predicted_centered + user_means.values.reshape(-1, 1)

# Clip to valid rating range [0.5, 5.0]
predicted_ratings = np.clip(predicted_ratings, 0.5, 5.0)

# Convert to DataFrame with same index/columns as original
pred_df = pd.DataFrame(predicted_ratings, index=rating_matrix.index, columns=rating_matrix.columns)

print(f"Predicted ratings matrix shape: {pred_df.shape}")
print(f"\nSample predictions for User 1 (first 5 movies):")
print(pred_df.iloc[0, :5])

Predicted ratings matrix shape: (610, 9724)

Sample predictions for User 1 (first 5 movies):
movieId
1    4.306161
2    4.296397
3    4.405951
4    4.370563
5    4.342908
Name: 1, dtype: float64


### Step 5: Generate Top-N Recommendations

In [7]:
def recommend_svd(user_id, pred_df, rating_matrix, movies_df, top_n=10):
    """
    Recommend top-N movies for a user using SVD-reconstructed ratings.
    Excludes movies the user has already rated.
    """
    # Get predicted ratings for this user
    user_preds = pred_df.loc[user_id]
    
    # Exclude already-rated movies
    already_rated = rating_matrix.loc[user_id].dropna().index
    user_preds = user_preds.drop(already_rated, errors='ignore')
    
    # Top-N by predicted rating
    top = user_preds.nlargest(top_n)
    
    results = pd.DataFrame({
        'movieId': top.index,
        'Predicted Rating': top.values.round(2)
    }).merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    
    return results[['title', 'genres', 'Predicted Rating']]

# Test for User 1
user1_rated = ratings[ratings['userId'] == 1].merge(movies, on='movieId')
print("User 1's top-rated movies:")
print(user1_rated.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 SVD Recommendations for User 1")
print("="*70)
recommend_svd(1, pred_df, rating_matrix, movies, top_n=10)

User 1's top-rated movies:
                                 title                             genres  rating
           Seven (a.k.a. Se7en) (1995)                   Mystery|Thriller     5.0
            Usual Suspects, The (1995)             Crime|Mystery|Thriller     5.0
                  Bottle Rocket (1996)     Adventure|Comedy|Crime|Romance     5.0
Dumb & Dumber (Dumb and Dumber) (1994)                   Adventure|Comedy     5.0
                  Billy Madison (1995)                             Comedy     5.0
                      Desperado (1995)             Action|Romance|Western     5.0
                 Canadian Bacon (1995)                         Comedy|War     5.0
                        Rob Roy (1995)           Action|Drama|Romance|War     5.0
                      Pinocchio (1940) Animation|Children|Fantasy|Musical     5.0
                      Tombstone (1993)               Action|Drama|Western     5.0

Top 10 SVD Recommendations for User 1


,title,genres,Predicted Rating
0,Blade Runner (1982),Action|Sci-Fi|Thriller,4.71
1,Ace Ventura: Pet Detective (1994),Comedy,4.68
2,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,4.66
3,"Christmas Story, A (1983)",Children|Comedy,4.66
4,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.62
5,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.62
6,Dead Poets Society (1989),Drama,4.59
7,Snatch (2000),Comedy|Crime|Thriller,4.59
8,"Godfather, The (1972)",Crime|Drama,4.59
9,"Producers, The (1968)",Comedy,4.58


In [8]:
print("="*70)
print("Top 10 SVD Recommendations for User 5")
print("="*70)
recommend_svd(5, pred_df, rating_matrix, movies, top_n=10)

Top 10 SVD Recommendations for User 5


,title,genres,Predicted Rating
0,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,3.93
1,Forrest Gump (1994),Comedy|Drama|Romance|War,3.91
2,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,3.79
3,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,3.78
4,Heat (1995),Action|Crime|Thriller,3.74
5,Airplane! (1980),Comedy,3.73
6,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,3.73
7,Taxi Driver (1976),Crime|Drama|Thriller,3.72
8,One Flew Over the Cuckoo's Nest (1975),Drama,3.72
9,"American President, The (1995)",Comedy|Drama|Romance,3.72


In [9]:
print("="*70)
print("Top 10 SVD Recommendations for User 100")
print("="*70)
recommend_svd(100, pred_df, rating_matrix, movies, top_n=10)

Top 10 SVD Recommendations for User 100


,title,genres,Predicted Rating
0,"Shawshank Redemption, The (1994)",Crime|Drama,4.11
1,"Fugitive, The (1993)",Thriller,4.08
2,"Mummy, The (1999)",Action|Adventure|Comedy|Fantasy|Horror|Thriller,4.08
3,Good Will Hunting (1997),Drama|Romance,4.08
4,Apollo 13 (1995),Adventure|Drama|IMAX,4.08
5,Rambo: First Blood Part II (1985),Action|Adventure|Thriller,4.07
6,Titanic (1997),Drama|Romance,4.07
7,Mission: Impossible II (2000),Action|Adventure|Thriller,4.06
8,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,4.06
9,Little Miss Sunshine (2006),Adventure|Comedy|Drama,4.05


### Step 6: Evaluate — RMSE, Precision@K, Recall@K

In [10]:
def evaluate_svd(ratings_df, movies_df, k_factors=50, K_rec=10, threshold=4.0):
    """
    Evaluate SVD-based recommendations with train/test split.
    """
    train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
    
    # Build matrix from train set
    train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
    train_means = train_matrix.mean(axis=1)
    train_centered = train_matrix.sub(train_means, axis=0).fillna(0)
    
    # SVD
    U, sigma, Vt = svds(train_centered.values, k=k_factors)
    sigma_diag = np.diag(sigma)
    pred_centered = U @ sigma_diag @ Vt
    pred_ratings = np.clip(pred_centered + train_means.values.reshape(-1, 1), 0.5, 5.0)
    train_pred_df = pd.DataFrame(pred_ratings, index=train_matrix.index, columns=train_matrix.columns)
    
    # --- RMSE ---
    test_in_train = test_df[test_df['userId'].isin(train_matrix.index)]
    actuals, preds = [], []
    for _, row in test_in_train.iterrows():
        uid, mid, actual = int(row['userId']), int(row['movieId']), row['rating']
        if mid in train_pred_df.columns:
            preds.append(train_pred_df.loc[uid, mid])
            actuals.append(actual)
    
    rmse = np.sqrt(mean_squared_error(actuals, preds)) if actuals else float('nan')
    
    # --- Precision@K and Recall@K ---
    precisions, recalls = [], []
    for uid in test_df['userId'].unique():
        if uid not in train_matrix.index:
            continue
        
        user_test = test_df[(test_df['userId'] == uid) & (test_df['rating'] >= threshold)]
        relevant = set(user_test['movieId'])
        if not relevant:
            continue
        
        recs = recommend_svd(uid, train_pred_df, train_matrix, movies_df, top_n=K_rec)
        if recs.empty:
            continue
        rec_ids = set(recs.merge(movies_df, left_on='title', right_on='title')['movieId'])
        
        hits = rec_ids & relevant
        precisions.append(len(hits) / K_rec)
        recalls.append(len(hits) / len(relevant))
    
    return {
        'RMSE': rmse,
        'Precision@K': np.mean(precisions) if precisions else 0,
        'Recall@K': np.mean(recalls) if recalls else 0,
        'RMSE_samples': len(actuals),
        'P/R_users': len(precisions)
    }

print("Evaluating SVD (k=50)...")
results = evaluate_svd(ratings, movies, k_factors=50, K_rec=10)
print(f"\nRMSE: {results['RMSE']:.4f} (on {results['RMSE_samples']} predictions)")
print(f"Precision@10: {results['Precision@K']:.4f}")
print(f"Recall@10: {results['Recall@K']:.4f}")
print(f"Evaluated users: {results['P/R_users']}")

Evaluating SVD (k=50)...

RMSE: 0.9304 (on 19355 predictions)
Precision@10: 0.1235
Recall@10: 0.1184
Evaluated users: 599


### Step 7: Compare SVD vs User-Based CF vs Item-Based CF

We test different numbers of latent factors and compare.

In [11]:
print("SVD performance across different k (latent factors):\n")
print(f"{'k':>4s} | {'RMSE':>8s} | {'Prec@10':>8s} | {'Recall@10':>9s}")
print("-" * 40)

for k in [10, 20, 50, 100]:
    res = evaluate_svd(ratings, movies, k_factors=k, K_rec=10)
    print(f"{k:>4d} | {res['RMSE']:>8.4f} | {res['Precision@K']:>8.4f} | {res['Recall@K']:>9.4f}")

print("\n" + "="*60)
print("Compare these results with Task 3 (User-Based CF) and Task 4")
print("(Item-Based CF) evaluation outputs.")
print("\nSVD typically achieves lower RMSE because it captures latent")
print("patterns across the entire matrix, while neighborhood methods")
print("rely on local similarity which can be noisy in sparse data.")
print("="*60)

SVD performance across different k (latent factors):

   k |     RMSE |  Prec@10 | Recall@10
----------------------------------------
  10 |   0.9184 |   0.1252 |    0.0978
  20 |   0.9207 |   0.1344 |    0.1146
  50 |   0.9304 |   0.1235 |    0.1184
 100 |   0.9396 |   0.0881 |    0.0999

Compare these results with Task 3 (User-Based CF) and Task 4
(Item-Based CF) evaluation outputs.

SVD typically achieves lower RMSE because it captures latent
patterns across the entire matrix, while neighborhood methods
rely on local similarity which can be noisy in sparse data.
